# Data quality

Runs the DQ rules from `data/dq-rules` over each silver stage table.

- **Hard** failures go to `quarantine.<entity>`.
- **Soft** failures stay in `silver.<entity>` with the rule IDs in `_dq_soft_failed_rule_ids`.
- Each run's violation counts go to `audit.silver_dq_audit`.

Run notebook 02 first: the `_raw` columns for the 'could not parse' rules come from it.

In [1]:
import sys
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from pipeline.session import get_spark_session
from pipeline.catalog import ensure_schema_exists, get_table_fqn, assert_table_exists
from pipeline.audit import AuditRun, combine_sinks, make_delta_sink
from pipeline.errors import step
from pipeline.events import EventBus
from pipeline.log import make_log_subscriber
from pipeline.reader import read_table
from pipeline.writer import replace_table
from silver.data_quality.definitions import RULES, rules_for
from silver.data_quality.evaluate import apply_data_quality, overall_score

spark = get_spark_session('data_quality_demo')


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 21:21:12 WARN Utils: Your hostname, DP41ZL4, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 21:21:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/charlescarpenter/venvs/mercury-delta/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/charlescarpenter/.ivy2.5.2/cache
The jars for the packages stored in: /home/charlescarpenter/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dep

In [2]:
import logging

logging.basicConfig(format='%(asctime)s %(levelname)s %(message)s', level=logging.INFO)

bus = EventBus()
bus.subscribe(make_log_subscriber(logging.getLogger('pipeline')))


1

In [3]:
CATALOG = 'spark_catalog'
STAGE_SCHEMA = 'silver_stage'
TRUSTED_SCHEMA, QUARANTINE_SCHEMA, AUDIT_SCHEMA_NAME = 'silver', 'quarantine', 'audit'
DQ_AUDIT_TABLE = f'{CATALOG}.{AUDIT_SCHEMA_NAME}.silver_dq_audit'

for schema in (TRUSTED_SCHEMA, QUARANTINE_SCHEMA, AUDIT_SCHEMA_NAME):
    ensure_schema_exists(spark, CATALOG, schema)

DQ_ENTITIES = tuple(RULES)
for entity in DQ_ENTITIES:
    rules = rules_for(entity)
    hard = sum(rule.is_hard for rule in rules)
    print(f'{entity:17} {len(rules):3} rules  hard={hard}  soft={len(rules) - hard}')


customer           10 rules  hard=4  soft=6
address            10 rules  hard=4  soft=6
category            7 rules  hard=2  soft=5
product             9 rules  hard=4  soft=5
product_category    8 rules  hard=3  soft=5
order              11 rules  hard=6  soft=5
order_item          9 rules  hard=6  soft=3
payment            11 rules  hard=7  soft=4
return             10 rules  hard=7  soft=3
return_item        10 rules  hard=7  soft=3


In [4]:
records = []
# Each run goes to the in-memory list (for the summary below) and to the audit table
audit_sink = combine_sinks(records.append, make_delta_sink(spark, DQ_AUDIT_TABLE))


def run_data_quality(entity: str) -> None:
    stage_table = get_table_fqn(CATALOG, STAGE_SCHEMA, entity)
    trusted_table = get_table_fqn(CATALOG, TRUSTED_SCHEMA, entity)
    quarantine_table = get_table_fqn(CATALOG, QUARANTINE_SCHEMA, entity)
    with AuditRun('silver_data_quality', entity, trusted_table, audit_sink, bus=bus) as run:
        with step('load_stage', bus=bus, entity=entity):
            assert_table_exists(spark, stage_table)
            df = read_table(spark, stage_table)
            run.add('stage_table', stage_table)

        with step('apply_rules', bus=bus, entity=entity):
            outcome = apply_data_quality(df, rules_for(entity))
            for name, value in outcome.metrics.items():
                run.add(name, value)

        with step('write_trusted', bus=bus, entity=entity):
            replace_table(outcome.trusted, trusted_table)

        with step('write_quarantine', bus=bus, entity=entity):
            replace_table(outcome.quarantine, quarantine_table)


In [5]:
for entity in DQ_ENTITIES:
    try:
        run_data_quality(entity)
    except Exception as exc:
        print(f'{entity}: failed: {exc}')

for record in records:
    m = record.metrics
    print(
        f"{record.entity:17} {record.status:8} score={m.get('dq_score')}  "
        f"rows={m.get('dq_row_count')}  trusted={m.get('dq_trusted_row_count')}  "
        f"quarantined={m.get('dq_quarantine_row_count')}  soft_warnings={m.get('dq_soft_warning_row_count')}"
        + (f"  error={record.error_message}" if record.error_message else '')
    )

succeeded = [record.metrics for record in records if record.status == 'success']
print(f'\nOverall data quality score: {overall_score(succeeded)}')


2026-10-11 21:21:20,921 INFO silver_data_quality customer run 1aab9336-eec2-416d-b376-3ef70f69d2d6 started
2026-10-11 21:21:21,912 INFO step 'load_stage' (entity=customer) completed in 0.99s
2026-10-11 21:21:21,934 ERROR step 'apply_rules' (entity=customer) failed: DataValidationError: Missing columns: date_of_birth_raw, marketing_opt_in_raw, phone_raw
2026-10-11 21:21:29,755 ERROR silver_data_quality customer run 1aab9336-eec2-416d-b376-3ef70f69d2d6 failed: Missing columns: date_of_birth_raw, marketing_opt_in_raw, phone_raw
2026-10-11 21:21:29,756 INFO silver_data_quality address run 7f871a3a-d3cb-409b-affd-6669dd98a3a7 started
2026-10-11 21:21:29,920 INFO step 'load_stage' (entity=address) completed in 0.16s
2026-10-11 21:21:29,923 ERROR step 'apply_rules' (entity=address) failed: DataValidationError: Missing columns: country_code_raw, is_primary_raw


customer: failed: Missing columns: date_of_birth_raw, marketing_opt_in_raw, phone_raw


2026-10-11 21:21:31,278 ERROR silver_data_quality address run 7f871a3a-d3cb-409b-affd-6669dd98a3a7 failed: Missing columns: country_code_raw, is_primary_raw
2026-10-11 21:21:31,279 INFO silver_data_quality category run 63d2f557-1014-4435-81d6-b2837a5d0776 started
2026-10-11 21:21:31,374 INFO step 'load_stage' (entity=category) completed in 0.09s
2026-10-11 21:21:31,377 ERROR step 'apply_rules' (entity=category) failed: DataValidationError: Missing columns: is_active_raw


address: failed: Missing columns: country_code_raw, is_primary_raw


2026-10-11 21:21:32,497 ERROR silver_data_quality category run 63d2f557-1014-4435-81d6-b2837a5d0776 failed: Missing columns: is_active_raw
2026-10-11 21:21:32,498 INFO silver_data_quality product run 6465ed32-3ca6-49cf-9b32-2abc1203ab18 started
2026-10-11 21:21:32,603 INFO step 'load_stage' (entity=product) completed in 0.11s
2026-10-11 21:21:32,608 ERROR step 'apply_rules' (entity=product) failed: DataValidationError: Missing columns: is_active_raw


...

product: failed: Missing columns: is_active_raw


2026-10-11 21:21:34,842 ERROR silver_data_quality product_category run e555c08b-f7cc-4dd9-87a1-b2c91c3c590f failed: Missing columns: active_from_raw, active_to_raw, is_primary_raw
2026-10-11 21:21:34,843 INFO silver_data_quality order run 10be0d82-23b2-45e5-b7ce-5a13a0cf6c74 started
2026-10-11 21:21:34,929 INFO step 'load_stage' (entity=order) completed in 0.09s


product_category: failed: Missing columns: active_from_raw, active_to_raw, is_primary_raw


2026-10-11 21:21:37,161 INFO step 'apply_rules' (entity=order) completed in 2.23s
2026-10-11 21:21:39,134 INFO step 'write_trusted' (entity=order) completed in 1.97s
2026-10-11 21:21:39,863 INFO step 'write_quarantine' (entity=order) completed in 0.73s
2026-10-11 21:21:40,873 INFO silver_data_quality order run 10be0d82-23b2-45e5-b7ce-5a13a0cf6c74 succeeded: {'stage_table': 'silver_stage.order', 'dq_row_count': 1000, 'dq_quarantine_row_count': 0, 'dq_soft_warning_row_count': 0, 'dq_clean_row_count': 1000, 'violations_ORD-001': 0, 'violations_ORD-002': 0, 'violations_ORD-003': 0, 'violations_ORD-004': 0, 'violations_ORD-005': 0, 'violations_ORD-006': 0, 'violations_ORD-007': 0, 'violations_ORD-008': 0, 'violations_ORD-009': 0, 'violations_ORD-010': 0, 'violations_ORD-012': 0, 'dq_trusted_row_count': 1000, 'dq_score': 100.0}
2026-10-11 21:21:40,874 INFO silver_data_quality order_item run b7b9e1ea-b16c-4389-b516-67d29697ec11 started
2026-10-11 21:21:40,946 INFO step 'load_stage' (entity=or

customer          failed   score=None  rows=None  trusted=None  quarantined=None  soft_warnings=None  error=Missing columns: date_of_birth_raw, marketing_opt_in_raw, phone_raw
address           failed   score=None  rows=None  trusted=None  quarantined=None  soft_warnings=None  error=Missing columns: country_code_raw, is_primary_raw
category          failed   score=None  rows=None  trusted=None  quarantined=None  soft_warnings=None  error=Missing columns: is_active_raw
product           failed   score=None  rows=None  trusted=None  quarantined=None  soft_warnings=None  error=Missing columns: is_active_raw
product_category  failed   score=None  rows=None  trusted=None  quarantined=None  soft_warnings=None  error=Missing columns: active_from_raw, active_to_raw, is_primary_raw
order             success  score=100.0  rows=1000  trusted=1000  quarantined=0  soft_warnings=0
order_item        success  score=100.0  rows=3096  trusted=3096  quarantined=0  soft_warnings=0
payment           succes

In [6]:
import pandas as pd

# Every rule with its violation count, across the entities in this run
violations = [
    {'entity': record.entity, 'rule_id': name.removeprefix('violations_'), 'rows_failing': value}
    for record in records if record.status == 'success'
    for name, value in record.metrics.items() if name.startswith('violations_')
]
violation_summary = pd.DataFrame(violations).sort_values(['entity', 'rule_id'])
display(violation_summary[violation_summary['rows_failing'] > 0])


,entity,rule_id,rows_failing


In [7]:
from IPython.display import display

# Quarantined rows keep the rule IDs that failed, so the reason is visible next to the data
display(read_table(spark, get_table_fqn(CATALOG, QUARANTINE_SCHEMA, 'order')).toPandas())


,source_record_id,order_id,customer_id,billing_address_id,shipping_address_id,order_timestamp,channel,order_status,currency_code,subtotal_amount,...,_ingested_at,_raw_record_payload,_source_system,_source_file_path,_source_file_name,_source_batch_id,_pipeline_run_id,_ingested_timestamp,_dq_hard_failed_rule_ids,_dq_soft_failed_rule_ids
